# Module 6.3 — Insight Narrative and Business Communication
### Reference & live-demo notebook — AJEBO Finance MFB Home Loan Applications

**Publica Academy Data Analysis Programme · Module 6, Week 7 (Data Storytelling and Visualisation)**

Modules 6.1 and 6.2 gave you a clean dataset and honest charts. This notebook is about the last,
easiest-to-skip step: turning a chart into a sentence someone can act on. Every claim made in this
notebook is computed from the actual cleaned dataset below it — nothing here is asserted without
its source calculation shown, which is itself the discipline this topic teaches.

**Learning outcome:** structure findings as a narrative (context, finding, implication,
recommendation), write insight statements that pass the "so what" test, and adapt the same
analysis for a technical and a non-technical audience.

**Dataset:** `ajebo_finance_loan_applications_CLEANED.csv` — the 356-row output of the Module 6.1
cleaning pipeline.

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('ajebo_finance_loan_applications_CLEANED.csv', parse_dates=['Application_Date'])

GOOD = '#028090'
ACCENT = '#02C39A'
MUTED = '#B7C4C4'
sns.set_style('white')
plt.rcParams['font.size'] = 11

df.shape


(356, 18)

---
## 1. Why narrative matters

A chart answers "what happened." It does not answer "why does this matter" or "what should we do
about it." The table below is the core distinction this whole topic teaches:

| | A finding | An insight |
|---|---|---|
| What it is | An observation about the data | An observation *plus* its business implication |
| What the reader does with it | Nods, moves on | Has something to act on, or a specific question to investigate |

Charts produce findings. A narrative turns findings into insights. Everything below is about that
conversion — and about never converting a finding into an insight you can't actually verify.

---
## 2. The "so what" test

Before writing any insight sentence, ask: *if a busy manager read only this sentence, could they
reasonably reply "so what?"* If yes, it isn't finished.

First, let's compute the real numbers behind four findings from Modules 6.1-6.2, so every
insight built on them in this notebook is traceable back to a calculation, not a guess.

In [2]:
# Kano's approval rate and its sample size
branch_stats = df.groupby('Branch_Region').agg(
    n=('Application_ID', 'count'),
    approval_rate=('Loan_Status', lambda s: (s == 'Approved').mean())
).sort_values('approval_rate')

# 95% confidence interval for Kano specifically
kano = branch_stats.loc['Kano']
se = np.sqrt(kano.approval_rate * (1 - kano.approval_rate) / kano.n)
kano_ci = (kano.approval_rate - 1.96 * se, kano.approval_rate + 1.96 * se)

print(branch_stats.round(3))
print(f"\nKano 95% CI: {kano_ci[0]*100:.0f}% - {kano_ci[1]*100:.0f}%")


                 n  approval_rate
Branch_Region                    
Kano            32          0.500
Ibadan          41          0.732
Abuja           62          0.742
Port Harcourt   45          0.756
Lagos          109          0.771
Rural North     67          0.776

Kano 95% CI: 33% - 67%


In [3]:
# Income vs loan amount relationship
income_loan_corr = df['Monthly_Income_NGN'].corr(df['Loan_Amount_NGN'])

# Self-employed vs salaried: loan amount and income
sub = df[df['Employment_Type'].isin(['Salaried', 'Self-Employed'])]
loan_by_emp = sub.groupby('Employment_Type')['Loan_Amount_NGN'].mean()
income_by_emp = sub.groupby('Employment_Type')['Monthly_Income_NGN'].mean()
loan_gap_pct = (loan_by_emp['Self-Employed'] - loan_by_emp['Salaried']) / loan_by_emp['Salaried'] * 100
income_gap_pct = (income_by_emp['Self-Employed'] - income_by_emp['Salaried']) / income_by_emp['Salaried'] * 100

# Education vs approval
edu_appr = df.groupby('Education')['Loan_Status'].apply(lambda s: (s == 'Approved').mean())
edu_n = df['Education'].value_counts()

print(f"Income-loan correlation: r = {income_loan_corr:.2f}")
print(f"Self-employed loans are {loan_gap_pct:.0f}% larger on average (NGN {loan_by_emp['Self-Employed']:,.0f} vs NGN {loan_by_emp['Salaried']:,.0f})")
print(f"Self-employed income is {income_gap_pct:.0f}% higher on average (NGN {income_by_emp['Self-Employed']:,.0f} vs NGN {income_by_emp['Salaried']:,.0f})")
print(f"Graduate approval rate: {edu_appr['Graduate']*100:.1f}% (n={edu_n['Graduate']})")
print(f"Not Graduate approval rate: {edu_appr['Not Graduate']*100:.1f}% (n={edu_n['Not Graduate']})")


Income-loan correlation: r = 0.72
Self-employed loans are 22% larger on average (NGN 3,822,128 vs NGN 3,129,732)
Self-employed income is 27% higher on average (NGN 279,963 vs NGN 221,018)
Graduate approval rate: 77.8% (n=261)
Not Graduate approval rate: 62.1% (n=95)


**Before-and-after, using the numbers just computed:**

| Finding (fails the test) | Insight (passes the test) |
|---|---|
| "Income and loan amount are correlated." | "Loan officers can reasonably estimate a request's likely size from income alone (r=0.72), which means income verification — not loan-amount justification — is where fraud risk concentrates." |
| "Self-employed applicants request larger loans on average." | "Self-employed applicants request 22% larger loans on average, but their incomes are also more volatile — underwriting for this segment may need a different verification standard, not just a different loan cap." |
| "Graduates are approved at a higher rate than non-graduates." | "The 16-point approval gap between graduates and non-graduates (77.8% vs 62.1%) is large enough, even on a modest sample of 95, to be worth checking against actual repayment data — is this a fair proxy for risk, or a bias worth removing?" |

Notice every insight names a number **and** an implication or next question — never just a
better-phrased version of the same finding.

---
### A practical tool: a lightweight insight-quality checker

This won't replace your own judgement, but it's a useful first pass: a function that flags common
"so what"-test failures — vague hedge words, no number, and no implication/action language.

In [4]:
import re

VAGUE_WORDS = ['interesting', 'some', 'various', 'a lot', 'notable', 'significant amount',
               'quite', 'somewhat', 'appears to', 'seems to', 'may or may not']
IMPLICATION_WORDS = ['means', 'suggests', 'recommend', 'because', 'should', 'risk', 'worth',
                     'implies', 'therefore', 'which means', 'next step', 'consider']

def check_insight(sentence):
    flags = []
    has_number = bool(re.search(r'\d', sentence))
    has_implication = any(w in sentence.lower() for w in IMPLICATION_WORDS)
    vague_hits = [w for w in VAGUE_WORDS if w in sentence.lower()]

    if not has_number:
        flags.append('No number present — is this a specific finding or a vague impression?')
    if not has_implication:
        flags.append('No implication/action language found — what should the reader DO with this?')
    if vague_hits:
        flags.append(f'Vague language found: {vague_hits} — replace with a specific figure or comparison.')

    verdict = 'PASSES basic checks' if not flags else 'NEEDS WORK'
    print(f'"{sentence}"\n  -> {verdict}')
    for f in flags:
        print(f'     - {f}')
    print()

check_insight("The data shows some interesting patterns in loan amounts.")
check_insight("Loan size scales predictably with income (r=0.72), so income verification, not loan-amount justification, is where fraud risk concentrates.")


"The data shows some interesting patterns in loan amounts."
  -> NEEDS WORK
     - No number present — is this a specific finding or a vague impression?
     - No implication/action language found — what should the reader DO with this?
     - Vague language found: ['interesting', 'some'] — replace with a specific figure or comparison.

"Loan size scales predictably with income (r=0.72), so income verification, not loan-amount justification, is where fraud risk concentrates."
  -> PASSES basic checks



Notice the checker correctly flags the vague sentence on three counts, and passes the sharp one
cleanly. **It is a first pass, not a replacement for judgement** — a sentence can contain a number
and an implication word and still be wrong or unverified, which is exactly what Section 5's
red-pen skill (Module 6.5) exists to catch. Try it on your own draft insights below.

In [5]:
# Try it yourself - edit this sentence and re-run
check_insight("Kano is underperforming and should be investigated.")


"Kano is underperforming and should be investigated."
  -> NEEDS WORK
     - No number present — is this a specific finding or a vague impression?



**Look closely at that result.** The checker only flagged the missing number — it did *not*
flag "should be investigated" as vague, because "should" is in its implication-word list. But
this sentence is exactly the kind of weak recommendation Section 3 warns about: no owner, no
method, no way to know when it's resolved. **This is the checker's real limitation, demonstrated
live**: it can catch the absence of a number or an implication word, but it cannot judge whether
an implication is actually specific. That judgement call is still yours — the tool narrows what
to look for, it doesn't replace looking.

---
## 3. The narrative structure: Context → Finding → Implication → Recommendation

Use this four-part structure for any insight reaching a decision-maker. Let's build it for Kano,
assembling each part from the numbers computed in Section 2 — not from memory.

In [6]:
ibadan = branch_stats.loc['Ibadan']  # next-lowest branch, for comparison
gap_points = (ibadan.approval_rate - kano.approval_rate) * 100

narrative = f'''
CONTEXT
AJEBO's credit committee reviews branch performance quarterly and is deciding whether Kano's
lending criteria need to change.

FINDING
Kano's approval rate is {kano.approval_rate*100:.0f}%, the lowest of the six branches
(next-lowest is Ibadan at {ibadan.approval_rate*100:.0f}%, a {gap_points:.0f}-point gap).
Kano's sample is small ({kano.n:.0f} applications), so this comes with a wide 95% confidence
interval ({kano_ci[0]*100:.0f}%-{kano_ci[1]*100:.0f}%).

IMPLICATION
The gap could reflect a genuine difference in local applicant risk, an inconsistency in how loan
officers at that branch apply the criteria, or simply be noisy given the small sample. The
approval-rate figure alone cannot distinguish between these explanations.

RECOMMENDATION
Before changing Kano's lending policy: pull a manual sample of Kano's rejected applications and
compare stated rejection reasons against a comparably-sized branch (Ibadan, n={ibadan.n:.0f}) for
consistency. Revisit the approval rate once Kano's sample reaches roughly 60-80 applications,
where the confidence interval narrows meaningfully.
'''
print(narrative)



CONTEXT
AJEBO's credit committee reviews branch performance quarterly and is deciding whether Kano's
lending criteria need to change.

FINDING
Kano's approval rate is 50%, the lowest of the six branches
(next-lowest is Ibadan at 73%, a 23-point gap).
Kano's sample is small (32 applications), so this comes with a wide 95% confidence
interval (33%-67%).

IMPLICATION
The gap could reflect a genuine difference in local applicant risk, an inconsistency in how loan
officers at that branch apply the criteria, or simply be noisy given the small sample. The
approval-rate figure alone cannot distinguish between these explanations.

RECOMMENDATION
Before changing Kano's lending policy: pull a manual sample of Kano's rejected applications and
compare stated rejection reasons against a comparably-sized branch (Ibadan, n=41) for
consistency. Revisit the approval rate once Kano's sample reaches roughly 60-80 applications,
where the confidence interval narrows meaningfully.



Every number in that narrative — 50%, 73%, the 23-point gap, n=32, the 33-67% interval — came
from a pandas calculation two cells above, not from memory or an approximate recollection.
That traceability is the entire point of this exercise.

---
## 4. Before-and-after critique — weak vs sharp insight statements

**Example 1**

> **Weak:** "The data shows some interesting patterns in loan amounts."

*Diagnose it:* which patterns? Interesting to whom, for what decision? This sentence could
introduce almost any chart in the deck and still be technically true — which means it says
nothing.

> **Sharp:** "Loan size scales predictably with income (r=0.72) for both employment types, but
> self-employed applicants' larger average loans come with more income volatility — a segment
> worth a distinct underwriting check, not just a higher loan cap."

**Example 2**

> **Weak:** "Kano is underperforming and should be investigated."

*Diagnose it:* exactly the sentence the checker above just let slip through — "underperforming"
against what benchmark, "investigated" by whom, doing what, by when?

> **Sharp:** "Kano's 50% approval rate is 23 points below the next-lowest branch, but its small
> sample (n=32) means the true rate could plausibly be as high as 67%. Recommend a manual file
> review of Kano's last 20 rejections, compared against Ibadan's, before any policy change."

**The rule:** *a sharp insight names a number, a comparison, a mechanism or reason, and a next
step. A weak one can be true of almost any dataset and still say nothing.*

---
## 5. Adapting for technical and non-technical audiences

Same finding, same underlying number — different vocabulary and emphasis depending on the
reader. Let's verify the number once, then write both versions from it.

In [7]:
print(f"Verified once: income-loan correlation r = {income_loan_corr:.2f}")


Verified once: income-loan correlation r = 0.72


**For a technical audience** (a fellow analyst, a data science lead):

> "Monthly income and requested loan amount show a Pearson correlation of 0.72 across both
> employment types, suggesting income could serve as a reasonable single-variable proxy in a
> simple loan-sizing model, though the relationship is noisier at the high-income tail where
> self-employed applicants cluster."

**For a non-technical audience** (the credit committee, a branch manager):

> "Applicants who earn more tend to ask for bigger loans — this holds fairly consistently across
> salaried and self-employed applicants alike. That means income is a solid first check when a
> loan request looks unusually large for the applicant's stated earnings."

**What changed:** the statistic became a plain-language pattern, and the implication shifted from
a modelling suggestion to an operational one. **What did not change:** the underlying finding —
neither version overstates or understates what r=0.72 actually shows.

---
## 6. Practical activity — write the narrative

Choose **two** of the five findings below. For each: verify its number with a quick calculation
(cells are scaffolded for you), then write a full Context → Finding → Implication →
Recommendation narrative, plus a one-paragraph non-technical version.

1. Kano's approval rate and small sample size (numbers already verified above).
2. The income-to-loan-amount relationship (numbers already verified above).
3. The graduate vs non-graduate approval gap (numbers already verified above).
4. Self-employed applicants' larger, more volatile loan requests (numbers already verified above).
5. The overall decline in monthly application volume since mid-2025 — **verify this one
   yourself** in the cell below before writing about it.

In [8]:
# Verify finding 5 yourself: has volume really declined, and by how much?
monthly_total = df.set_index('Application_Date').resample('ME').size()
monthly_total = monthly_total.iloc[:-1]  # exclude partial final month, as in Module 6.2

first_half = monthly_total.iloc[:len(monthly_total)//2].mean()
second_half = monthly_total.iloc[len(monthly_total)//2:].mean()
decline_pct = (first_half - second_half) / first_half * 100

print(f"First-half monthly average: {first_half:.1f} applications")
print(f"Second-half monthly average: {second_half:.1f} applications")
print(f"Decline: {decline_pct:.1f}%")


First-half monthly average: 21.1 applications
Second-half monthly average: 16.2 applications
Decline: 23.3%


**Write your two narratives here** (double-click this cell's neighbours below to edit them as
markdown, or replace with your own cells):

### Narrative 1: [choose a finding]

**Context:**

**Finding:**

**Implication:**

**Recommendation:**

**Non-technical version (one paragraph):**

---

### Narrative 2: [choose a finding]

**Context:**

**Finding:**

**Implication:**

**Recommendation:**

**Non-technical version (one paragraph):**

**Self-check before submitting:** run every insight sentence you wrote through `check_insight()`
from Section 2, then read each one again and ask whether the tool's PASS actually holds up to
your own judgement — remember its limitation demonstrated above.

In [9]:
# Paste your own insight sentences here to self-check them
check_insight("Replace this with your own sentence from Narrative 1.")


"Replace this with your own sentence from Narrative 1."
  -> NEEDS WORK
     - No implication/action language found — what should the reader DO with this?



---
## 7. Common mistakes

- **Restating the chart in words.** "The bar for Kano is shorter than the others" is a caption,
  not a narrative.
- **Overclaiming causation.** The data shows self-employed applicants request larger loans — it
  does not show *why*. Say "is associated with," not "causes," without evidence beyond a
  correlation.
- **A recommendation with no owner or method.** "We should look into this" fails the same test as
  a weak insight.
- **Skipping the small-sample caveat.** Kano's n=32 and the non-graduate group's n=95 both need
  their sample size stated before the implication, not buried in a footnote.
- **One version for every audience.** Sending the technical write-up to the credit committee (or
  vice versa) undermines trust in both directions.

---
## 8. Visualisation-to-narrative checklist

- [ ] Does the accompanying text name a specific number, not just "shows a pattern"?
- [ ] Does the insight pass the "so what" test?
- [ ] Is the sample size stated wherever it materially affects how much to trust the finding?
- [ ] Does the recommendation (if any) name a specific next action and how to know it worked?
- [ ] Have you avoided causal language where the evidence only supports a correlation?
- [ ] Is the version you're sending matched to its audience?

---
## 9. Knowledge check — 5 questions

1. What is the difference between a finding and an insight?
2. Rewrite this to pass the "so what" test: *"Self-employed applicants have higher average
   incomes."*
3. Name the four parts of the Context → Finding → Implication → Recommendation structure, in
   order.
4. Why is "we should investigate this further" not an acceptable recommendation on its own?
5. What should stay the same, and what should change, when adapting the same finding for a
   technical vs a non-technical audience?

<details><summary><b>Answer key</b></summary>

1. A finding is an observation about the data; an insight adds the business implication.
2. Many acceptable answers; a strong one names a number and an implication, e.g. "Self-employed
   applicants earn about 27% more on average than salaried applicants, but their incomes also
   vary more — a flat income threshold in underwriting may not fit this group as well as it does
   salaried applicants."
3. Context, Finding, Implication, Recommendation.
4. It names no owner, no method, and no way to know when the question is resolved.
5. The underlying facts stay the same; the vocabulary, statistical detail, and emphasised
   implication should change to match the audience.

</details>

---
## Next: Module 6.4 — GitHub for Portfolios

A sharp insight narrative is only useful if someone else can find it. Module 6.4 turns this
week's cleaned data, honest charts, and written narratives into a documented project a future
employer can actually open.